In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.feature_engineering import prepare_features

In [2]:
import pandas as pd

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

train = pd.read_parquet(
    PROCESSED_DIR / "train_canonical.parquet"
)

X_test = pd.read_parquet(
    PROCESSED_DIR / "X_test_canonical.parquet"
)

X_train_features = prepare_features(
    train.drop(columns=["Цена"])
)

X_test_features = prepare_features(X_test)

In [3]:
from sklearn.model_selection import train_test_split

ID_COLUMN = "car_id"
TARGET_COLUMN = "Цена"

EXCLUDED_COLUMNS = [
    ID_COLUMN,
    "Предложение",
]

feature_columns = [
    column
    for column in X_train_features.columns
    if column not in EXCLUDED_COLUMNS
]

X = X_train_features[feature_columns].copy()
X_final_test = X_test_features[feature_columns].copy()
y = train[TARGET_COLUMN].copy()

assert list(X.columns) == list(X_final_test.columns)
assert len(X) == len(y) == 8340
assert "Предложение" not in X.columns
assert "car_id" not in X.columns

In [4]:
import numpy as np
import pandas as pd

price_bins = [
    0,
    5_000,
    10_000,
    20_000,
    30_000,
    50_000,
    75_000,
    100_000,
    200_000,
    np.inf,
]

price_labels = [
    "0–5k",
    "5k–10k",
    "10k–20k",
    "20k–30k",
    "30k–50k",
    "50k–75k",
    "75k–100k",
    "100k–200k",
    "200k+",
]

Создадим бины target для стратификации:

In [5]:
target_bins = pd.qcut(
    y,
    q=10,
    duplicates="drop",
)

print(target_bins.value_counts().sort_index())

Цена
(899.999, 11999.0]      846
(11999.0, 16990.0]      862
(16990.0, 21494.9]      794
(21494.9, 24999.0]      843
(24999.0, 29620.0]      825
(29620.0, 33990.0]      835
(33990.0, 39990.0]      901
(39990.0, 48888.0]      768
(48888.0, 64990.0]      839
(64990.0, 1500000.0]    827
Name: count, dtype: int64


In [6]:
X_train_split, X_valid_split, y_train_split, y_valid_split = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=target_bins,
)

print("Train:", X_train_split.shape, y_train_split.shape)
print("Validation:", X_valid_split.shape, y_valid_split.shape)

Train: (6672, 27) (6672,)
Validation: (1668, 27) (1668,)


Проверка, что распределения цены в частях действительно похожи:

In [7]:
split_target_report = pd.DataFrame(
    {
        "train": y_train_split.describe(),
        "validation": y_valid_split.describe(),
        "relative_difference_pct": (
            (
                y_valid_split.describe()
                - y_train_split.describe()
            )
            / y_train_split.describe()
            * 100
        ),
    }
).round(2)

display(split_target_report)

,train,validation,relative_difference_pct
count,6672.00,1668.00,-75.00
mean,37312.61,36384.42,-2.49
std,38272.54,30182.03,-21.14
min,900.00,1200.00,33.33
25%,19472.50,19369.75,-0.53
50%,29645.00,29565.00,-0.27
75%,43990.00,43990.00,0.00
max,1500000.00,339900.00,-77.34


Просмотр доли  ценовых сегментов

In [8]:
def get_price_segment_distribution(target: pd.Series) -> pd.Series:
    segments = pd.cut(
        target,
        bins=price_bins,
        labels=price_labels,
        include_lowest=True,
    )

    return (
        segments
        .value_counts(normalize=True)
        .reindex(price_labels, fill_value=0)
        .mul(100)
    )

segment_split_report = pd.DataFrame(
    {
        "train_share_pct": get_price_segment_distribution(y_train_split),
        "validation_share_pct": get_price_segment_distribution(y_valid_split),
    }
).round(2)

segment_split_report["difference_pct_points"] = (
    segment_split_report["validation_share_pct"]
    - segment_split_report["train_share_pct"]
).round(2)

display(segment_split_report)

,train_share_pct,validation_share_pct,difference_pct_points
Цена,,,
0–5k,0.49,0.72,0.23
5k–10k,6.06,6.89,0.83
10k–20k,21.84,20.56,-1.28
20k–30k,24.88,25.00,0.12
30k–50k,28.63,28.48,-0.15
50k–75k,11.15,11.51,0.36
75k–100k,3.75,3.90,0.15
100k–200k,2.46,2.28,-0.18
200k+,0.75,0.66,-0.09


In [9]:
from sklearn.metrics import mean_absolute_percentage_error

def mape_percent(y_true, y_pred) -> float:
    return mean_absolute_percentage_error(y_true, y_pred) * 100


constant_predictions = {
    "mean_train_price": y_train_split.mean(),
    "median_train_price": y_train_split.median(),
}

baseline_results = []

for baseline_name, constant_value in constant_predictions.items():
    y_pred_train = np.full(
        shape=len(y_train_split),
        fill_value=constant_value,
        dtype=float,
    )
    
    y_pred_valid = np.full(
        shape=len(y_valid_split),
        fill_value=constant_value,
        dtype=float,
    )

    baseline_results.append(
        {
            "model": baseline_name,
            "constant_prediction": round(constant_value, 2),
            "train_mape_pct": round(
                mape_percent(y_train_split, y_pred_train),
                3,
            ),
            "validation_mape_pct": round(
                mape_percent(y_valid_split, y_pred_valid),
                3,
            ),
        }
    )

baseline_results = pd.DataFrame(baseline_results)
display(baseline_results)

,model,constant_prediction,train_mape_pct,validation_mape_pct
0,mean_train_price,37312.61,82.176,84.879
1,median_train_price,29645.00,61.977,64.307


In [10]:
import numpy as np


def weighted_median(values, weights):
    values = np.asarray(values)
    weights = np.asarray(weights)

    sort_idx = np.argsort(values)

    sorted_values = values[sort_idx]
    sorted_weights = weights[sort_idx]

    cumulative_weights = np.cumsum(sorted_weights)
    cutoff = sorted_weights.sum() / 2

    return sorted_values[np.searchsorted(cumulative_weights, cutoff)]


mape_optimal_constant = weighted_median(
    values=y_train_split.to_numpy(),
    weights=1 / y_train_split.to_numpy(),
)

y_pred_train = np.full(
    len(y_train_split),
    mape_optimal_constant,
)

y_pred_valid = np.full(
    len(y_valid_split),
    mape_optimal_constant,
)

weighted_median_result = pd.DataFrame(
    {
        "model": ["mape_weighted_median"],
        "constant_prediction": [round(mape_optimal_constant, 2)],
        "train_mape_pct": [
            round(mape_percent(y_train_split, y_pred_train), 3)
        ],
        "validation_mape_pct": [
            round(mape_percent(y_valid_split, y_pred_valid), 3)
        ],
    }
)

baseline_results = pd.concat(
    [baseline_results, weighted_median_result],
    ignore_index=True,
)

display(baseline_results)

,model,constant_prediction,train_mape_pct,validation_mape_pct
0,mean_train_price,37312.61,82.176,84.879
1,median_train_price,29645.00,61.977,64.307
2,mape_weighted_median,18999.00,50.263,51.724


Код для Ridge baseline

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [12]:
RAW_DUPLICATE_NUMERIC_COLUMNS = [
    "Пробег",
    "Расход",
    "Количество цилиндров",
    "Двери",
    "Количество кресел",
]

BASELINE_EXCLUDED_COLUMNS = [
    "car_id",
    "Предложение",
] + RAW_DUPLICATE_NUMERIC_COLUMNS

ridge_feature_columns = [
    column
    for column in X_train_split.columns
    if column not in BASELINE_EXCLUDED_COLUMNS
]

numeric_columns = [
    "Год выпуска",
    "Оценка эксперта",
    "Количество владельцев",
    "Пробег_число",
    "Расход_л_на_100км",
    "Двигатель_цилиндры",
    "Двигатель_объём_л",
    "Двери_число",
    "Кресла_число",
]

categorical_columns = [
    column
    for column in ridge_feature_columns
    if column not in numeric_columns
]

print("Числовые признаки:")
print(numeric_columns)

print("\nКатегориальные признаки:")
print(categorical_columns)

assert set(numeric_columns).issubset(ridge_feature_columns)
assert set(numeric_columns + categorical_columns) == set(ridge_feature_columns)

Числовые признаки:
['Год выпуска', 'Оценка эксперта', 'Количество владельцев', 'Пробег_число', 'Расход_л_на_100км', 'Двигатель_цилиндры', 'Двигатель_объём_л', 'Двери_число', 'Кресла_число']

Категориальные признаки:
['Бренд', 'Модель', 'Тип машины', 'Полное название', 'Исползование', 'КПП', 'Двигатель', 'Привод', 'Топливо', 'Цвет', 'Локация', 'Тип кузова', 'Штат']


Подготовим матрицы:

In [13]:
X_train_ridge = X_train_split[ridge_feature_columns].copy()
X_valid_ridge = X_valid_split[ridge_feature_columns].copy()

# Делаем категориальные пропуски совместимыми с sklearn.
for column in categorical_columns:
    X_train_ridge[column] = (
        X_train_ridge[column]
        .astype("object")
        .where(X_train_ridge[column].notna(), np.nan)
    )

    X_valid_ridge[column] = (
        X_valid_ridge[column]
        .astype("object")
        .where(X_valid_ridge[column].notna(), np.nan)
    )

Создадим pipeline:

In [14]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="__MISSING__",
            ),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_columns),
        ("cat", categorical_pipeline, categorical_columns),
    ],
    remainder="drop",
)

ridge_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            Ridge(
                alpha=10.0,
                solver="lsqr",
            ),
        ),
    ]
)

Обучение и оценка:

In [15]:
ridge_model.fit(X_train_ridge, y_train_split)

y_pred_train = ridge_model.predict(X_train_ridge)
y_pred_valid = ridge_model.predict(X_valid_ridge)

# Цена не должна быть отрицательной.
y_pred_train_clipped = np.maximum(y_pred_train, 1)
y_pred_valid_clipped = np.maximum(y_pred_valid, 1)

ridge_result = pd.DataFrame(
    {
        "model": ["ridge_ohe_alpha_10"],
        "train_mape_pct": [
            round(mape_percent(y_train_split, y_pred_train_clipped), 3)
        ],
        "validation_mape_pct": [
            round(mape_percent(y_valid_split, y_pred_valid_clipped), 3)
        ],
        "train_mae": [
            round(mean_absolute_error(y_train_split, y_pred_train_clipped), 2)
        ],
        "validation_mae": [
            round(mean_absolute_error(y_valid_split, y_pred_valid_clipped), 2)
        ],
        "negative_train_predictions": [
            int((y_pred_train <= 0).sum())
        ],
        "negative_validation_predictions": [
            int((y_pred_valid <= 0).sum())
        ],
    }
)

display(ridge_result)

,model,train_mape_pct,validation_mape_pct,train_mae,validation_mae,negative_train_predictions,negative_validation_predictions
0,ridge_ohe_alpha_10,26.204,30.523,7579.95,8677.4,131,39


In [16]:
transformed_feature_count = len(
    ridge_model.named_steps["preprocessor"].get_feature_names_out()
)

print("Признаков после преобразований:", transformed_feature_count)

Признаков после преобразований: 6779


разберём, где Ridge ошибается: по ценовым сегментам и в какую сторону.

In [ ]:
valid_predictions = pd.DataFrame(
    {
        "y_true": y_valid_split.to_numpy(),
        "y_pred": y_pred_valid_clipped,
    },
    index=y_valid_split.index,
)

valid_predictions["absolute_error"] = (
    valid_predictions["y_true"] - valid_predictions["y_pred"]
).abs()

valid_predictions["ape_pct"] = (
    valid_predictions["absolute_error"]
    / valid_predictions["y_true"]
    * 100
)

valid_predictions["relative_error_pct"] = (
    (
        valid_predictions["y_pred"]
        - valid_predictions["y_true"]
    )
    / valid_predictions["y_true"]
    * 100
)

valid_predictions["price_segment"] = pd.cut(
    valid_predictions["y_true"],
    bins=price_bins,
    labels=price_labels,
    include_lowest=True,
)

ridge_segment_report = (
    valid_predictions
    .groupby("price_segment", observed=False)
    .agg(
        objects=("y_true", "size"),
        mean_true_price=("y_true", "mean"),
        mean_predicted_price=("y_pred", "mean"),
        mape_pct=("ape_pct", "mean"),
        mae=("absolute_error", "mean"),
        mean_relative_error_pct=("relative_error_pct", "mean"),
    )
    .round(2)
)

display(ridge_segment_report)

,objects,mean_true_price,mean_predicted_price,mape_pct,mae,mean_relative_error_pct
price_segment,,,,,,
0–5k,12,3763.25,6888.20,173.48,5693.52,99.95
5k–10k,115,8467.85,7674.33,75.08,6292.05,-12.76
10k–20k,343,15933.90,17994.22,45.25,7107.97,10.83
20k–30k,417,25682.30,27718.64,24.31,6171.90,8.12
30k–50k,475,39257.53,42170.46,18.89,7303.10,7.57
50k–75k,192,60311.62,60790.15,16.35,9954.85,0.54
75k–100k,65,84748.89,81021.64,17.59,14775.05,-4.24
100k–200k,38,128679.18,110416.20,22.51,31133.43,-12.25
200k+,11,260887.82,165082.56,38.84,104229.65,-35.25


самые тяжёлые ошибки:

In [18]:
worst_ridge_predictions = (
    valid_predictions
    .join(
        X_valid_split[
            [
                "Бренд",
                "Модель",
                "Год выпуска",
                "Пробег_число",
                "Полное название",
            ]
        ]
    )
    .sort_values("ape_pct", ascending=False)
)

display(worst_ridge_predictions.head(20))

,y_true,y_pred,absolute_error,ape_pct,relative_error_pct,price_segment,Бренд,Модель,Год выпуска,Пробег_число,Полное название
5189,3999,41646.718093,37647.718093,941.428309,941.428309,0–5k,BMW,320CI,2004.0,268205,2004 BMW 320CI
3118,1200,6894.586500,5694.586500,474.548875,474.548875,0–5k,MAZDA,6,2004.0,201123,2004 MAZDA 6 LUXURY SPORTS
997,9999,50137.755780,40138.755780,401.427701,401.427701,5k–10k,MERCEDES-BENZ,MB100,2002.0,50885,2002 MERCEDES-BENZ MB100 D (2.9)
1122,14999,71882.780599,56883.780599,379.250487,379.250487,10k–20k,MERCEDES-BENZ,ML500,2006.0,160277,2006 MERCEDES-BENZ ML500 LUXURY (4X4)
3324,8995,38057.778302,29062.778302,323.099258,323.099258,5k–10k,HOLDEN,CAPTIVA,2012.0,120500,2012 HOLDEN CAPTIVA 7 LX (4X4)
5573,18890,75684.135281,56794.135281,300.657148,300.657148,10k–20k,PORSCHE,CAYENNE,2007.0,206149,2007 PORSCHE CAYENNE S
4674,8999,34554.876599,25555.876599,283.985738,283.985738,5k–10k,CHRYSLER,GRAND,2002.0,154289,2002 CHRYSLER GRAND VOYAGER LIMITED
5994,7999,30057.442417,22058.442417,275.765001,275.765001,5k–10k,CHRYSLER,SEBRING,2009.0,152721,2009 CHRYSLER SEBRING JS LIMITED SEDAN 4DR SPT...
1180,22995,84545.290085,61550.290085,267.668146,267.668146,20k–30k,BMW,X6,2011.0,168282,2011 BMW X6 XDRIVE 50I
5114,8999,31356.339588,22357.339588,248.442489,248.442489,5k–10k,OPEL,ASTRA,2013.0,86852,2013 OPEL ASTRA 1.4


Следующий разумный эксперимент:

Ridge + тот же preprocessing
→ обучаемся на log1p(Цена)
→ возвращаем прогноз через expm1
→ считаем MAPE на исходной шкале цены

Интуитивно: модель будет думать не столько об абсолютной разнице в рублях, сколько о соотношении цен. Ошибка +10k для машины за 5k и за 100k перестанет восприниматься одинаково тяжёлой.

Но это гипотеза. Сравниваем только по validation MAPE.

эксперимент с log1p(Цена).

In [19]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.base import clone

In [20]:
ridge_log_target_model = TransformedTargetRegressor(
    regressor=clone(ridge_model),
    func=np.log1p,
    inverse_func=np.expm1,
    check_inverse=False,
)

ridge_log_target_model.fit(
    X_train_ridge,
    y_train_split,
)

y_pred_train_log = ridge_log_target_model.predict(X_train_ridge)
y_pred_valid_log = ridge_log_target_model.predict(X_valid_ridge)

# Защита от невозможных предсказаний цены.
y_pred_train_log_clipped = np.maximum(y_pred_train_log, 1)
y_pred_valid_log_clipped = np.maximum(y_pred_valid_log, 1)

Сравниваем результаты с обычным Ridge:

In [21]:
ridge_log_result = pd.DataFrame(
    {
        "model": ["ridge_ohe_log1p_target_alpha_10"],
        "train_mape_pct": [
            round(
                mape_percent(y_train_split, y_pred_train_log_clipped),
                3,
            )
        ],
        "validation_mape_pct": [
            round(
                mape_percent(y_valid_split, y_pred_valid_log_clipped),
                3,
            )
        ],
        "train_mae": [
            round(
                mean_absolute_error(
                    y_train_split,
                    y_pred_train_log_clipped,
                ),
                2,
            )
        ],
        "validation_mae": [
            round(
                mean_absolute_error(
                    y_valid_split,
                    y_pred_valid_log_clipped,
                ),
                2,
            )
        ],
        "negative_train_predictions": [
            int((y_pred_train_log <= 0).sum())
        ],
        "negative_validation_predictions": [
            int((y_pred_valid_log <= 0).sum())
        ],
    }
)

display(
    pd.concat(
        [ridge_result, ridge_log_result],
        ignore_index=True,
    )
)

,model,train_mape_pct,validation_mape_pct,train_mae,validation_mae,negative_train_predictions,negative_validation_predictions
0,ridge_ohe_alpha_10,26.204,30.523,7579.95,8677.40,131,39
1,ridge_ohe_log1p_target_alpha_10,13.390,16.085,5621.55,5962.48,0,0


сегментный анализ для log-модели, чтобы сравнить не только одну среднюю цифру:

In [22]:
valid_predictions_log = pd.DataFrame(
    {
        "y_true": y_valid_split.to_numpy(),
        "y_pred": y_pred_valid_log_clipped,
    },
    index=y_valid_split.index,
)

valid_predictions_log["absolute_error"] = (
    valid_predictions_log["y_true"]
    - valid_predictions_log["y_pred"]
).abs()

valid_predictions_log["ape_pct"] = (
    valid_predictions_log["absolute_error"]
    / valid_predictions_log["y_true"]
    * 100
)

valid_predictions_log["relative_error_pct"] = (
    (
        valid_predictions_log["y_pred"]
        - valid_predictions_log["y_true"]
    )
    / valid_predictions_log["y_true"]
    * 100
)

valid_predictions_log["price_segment"] = pd.cut(
    valid_predictions_log["y_true"],
    bins=price_bins,
    labels=price_labels,
    include_lowest=True,
)

ridge_log_segment_report = (
    valid_predictions_log
    .groupby("price_segment", observed=False)
    .agg(
        objects=("y_true", "size"),
        mape_pct=("ape_pct", "mean"),
        mae=("absolute_error", "mean"),
        mean_relative_error_pct=("relative_error_pct", "mean"),
    )
    .round(2)
)

display(ridge_log_segment_report)

,objects,mape_pct,mae,mean_relative_error_pct
price_segment,,,,
0–5k,12,121.67,3436.06,121.67
5k–10k,115,25.93,2170.42,21.94
10k–20k,343,16.61,2674.96,7.03
20k–30k,417,13.30,3429.58,1.44
30k–50k,475,12.73,4987.17,-0.30
50k–75k,192,14.26,8656.93,-4.58
75k–100k,65,16.27,13795.50,-5.43
100k–200k,38,22.25,30360.45,-13.97
200k+,11,41.81,111409.43,-41.81


Текущий лидер экспериментов:

Ridge + OneHotEncoder + log1p(target)
Validation MAPE: 16.085%

подберём alpha для Ridge. Мы меняем только силу регуляризации, чтобы понять, не слишком ли модель сейчас переобучается на тысячах OHE-признаков.

In [23]:
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor

alpha_values = [0.1, 1.0, 3.0, 10.0, 30.0, 100.0, 300.0]

ridge_alpha_results = []

for alpha in alpha_values:
    model = Pipeline(
        steps=[
            ("preprocessor", clone(preprocessor)),
            (
                "model",
                Ridge(
                    alpha=alpha,
                    solver="lsqr",
                ),
            ),
        ]
    )

    model_log_target = TransformedTargetRegressor(
        regressor=model,
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False,
    )

    model_log_target.fit(X_train_ridge, y_train_split)

    train_pred = np.maximum(
        model_log_target.predict(X_train_ridge),
        1,
    )

    valid_pred = np.maximum(
        model_log_target.predict(X_valid_ridge),
        1,
    )

    ridge_alpha_results.append(
        {
            "alpha": alpha,
            "train_mape_pct": round(
                mape_percent(y_train_split, train_pred),
                3,
            ),
            "validation_mape_pct": round(
                mape_percent(y_valid_split, valid_pred),
                3,
            ),
            "gap_pct_points": round(
                mape_percent(y_valid_split, valid_pred)
                - mape_percent(y_train_split, train_pred),
                3,
            ),
        }
    )

ridge_alpha_results = (
    pd.DataFrame(ridge_alpha_results)
    .sort_values("validation_mape_pct")
    .reset_index(drop=True)
)

display(ridge_alpha_results)

,alpha,train_mape_pct,validation_mape_pct,gap_pct_points
0,1.0,6.588,14.183,7.595
1,0.1,2.626,14.194,11.568
2,3.0,9.866,14.808,4.942
3,10.0,13.390,16.085,2.694
4,30.0,16.104,17.581,1.477
5,100.0,18.573,19.429,0.855
6,300.0,20.494,21.135,0.641


alpha=0.1 уже явно переобучается.

alpha=1.0 пока лидер по validation, но gap тоже заметный.

alpha=3.0 немного хуже по holdout, зато выглядит устойчивее.

# ============================================================
# Ridge: проверка alpha через Stratified K-Fold CV
# ============================================================

In [24]:
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

X_ridge_all = X[ridge_feature_columns].copy()

# Для SimpleImputer категориальные пропуски должны быть обычными np.nan.
for column in categorical_columns:
    X_ridge_all[column] = (
        X_ridge_all[column]
        .astype("object")
        .where(X_ridge_all[column].notna(), np.nan)
    )

# Бины используются только для равномерного распределения цен по фолдам.
cv_target_bins = pd.qcut(
    y,
    q=10,
    labels=False,
    duplicates="drop",
).to_numpy()

assert len(cv_target_bins) == len(y)
assert pd.Series(cv_target_bins).notna().all()

display(
    pd.Series(cv_target_bins, name="price_bin")
    .value_counts()
    .sort_index()
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

alpha_candidates = [0.1, 1.0, 3.0, 10.0]

cv_rows = []

for alpha in alpha_candidates:
    fold_scores = []

    for fold_number, (train_idx, valid_idx) in enumerate(
        cv.split(X_ridge_all, cv_target_bins),
        start=1,
        ):
        X_fold_train = X_ridge_all.iloc[train_idx]
        X_fold_valid = X_ridge_all.iloc[valid_idx]

        y_fold_train = y.iloc[train_idx]
        y_fold_valid = y.iloc[valid_idx]

        ridge_cv_model = Pipeline(
            steps=[
                ("preprocessor", clone(preprocessor)),
                (
                    "model",
                    Ridge(
                        alpha=alpha,
                        solver="lsqr",
                    ),
                ),
            ]
        )

        model_log_target = TransformedTargetRegressor(
            regressor=ridge_cv_model,
            func=np.log1p,
            inverse_func=np.expm1,
            check_inverse=False,
        )

        model_log_target.fit(X_fold_train, y_fold_train)

        fold_pred = np.maximum(
            model_log_target.predict(X_fold_valid),
            1,
        )

        fold_mape = mape_percent(
            y_fold_valid,
            fold_pred,
        )

        fold_scores.append(fold_mape)

        cv_rows.append(
            {
                "alpha": alpha,
                "fold": fold_number,
                "validation_mape_pct": round(fold_mape, 3),
            }
        )

cv_fold_results = pd.DataFrame(cv_rows)

ridge_cv_results = (
    cv_fold_results
    .groupby("alpha", as_index=False)
    .agg(
        cv_mape_mean_pct=("validation_mape_pct", "mean"),
        cv_mape_std_pct=("validation_mape_pct", "std"),
        best_fold_mape_pct=("validation_mape_pct", "min"),
        worst_fold_mape_pct=("validation_mape_pct", "max"),
    )
    .sort_values("cv_mape_mean_pct")
    .round(3)
    .reset_index(drop=True)
)

display(ridge_cv_results)
display(
    cv_fold_results
    .pivot(
        index="alpha",
        columns="fold",
        values="validation_mape_pct",
    )
)

price_bin
0    846
1    862
2    794
3    843
4    825
5    835
6    901
7    768
8    839
9    827
Name: count, dtype: int64

,alpha,cv_mape_mean_pct,cv_mape_std_pct,best_fold_mape_pct,worst_fold_mape_pct
0,0.1,14.469,0.243,14.170,14.822
1,1.0,14.503,0.363,14.025,14.975
2,3.0,15.125,0.453,14.460,15.648
3,10.0,16.283,0.520,15.474,16.773


fold,1,2,3,4,5
alpha,,,,,
0.1,14.822,14.417,14.369,14.567,14.170
1.0,14.975,14.025,14.307,14.698,14.509
3.0,15.648,14.460,14.923,15.335,15.261
10.0,16.773,15.474,16.070,16.511,16.585


Для линейного baseline фиксируем:

Ridge + OneHotEncoder + log1p(target)
alpha = 0.1
5-fold stratified CV MAPE = 14.469% ± 0.243

In [25]:
BEST_RIDGE_ALPHA = 0.1

In [26]:
from pathlib import Path
import pandas as pd
import numpy as np

REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

BEST_RIDGE_ALPHA = 0.1

Создаю и обучаю лучший Ridge так же, как в подборе alpha:

In [27]:
best_ridge_pipeline = Pipeline(
    steps=[
        ("preprocessor", clone(preprocessor)),
        (
            "model",
            Ridge(
                alpha=BEST_RIDGE_ALPHA,
                solver="lsqr",
            ),
        ),
    ]
)

best_ridge_model = TransformedTargetRegressor(
    regressor=best_ridge_pipeline,
    func=np.log1p,
    inverse_func=np.expm1,
    check_inverse=False,
)

best_ridge_model.fit(
    X_train_ridge,
    y_train_split,
)

ridge_valid_pred = np.maximum(
    best_ridge_model.predict(X_valid_ridge),
    1,
)

ridge_train_pred = np.maximum(
    best_ridge_model.predict(X_train_ridge),
    1,
)

результат:

In [28]:
best_ridge_holdout_result = pd.DataFrame(
    {
        "experiment_id": ["ridge_ohe_log1p_alpha_0_1_holdout_rs42"],
        "model": ["Ridge + OHE + log1p(target)"],
        "split": ["train_valid_80_20_stratified_qcut10_rs42"],
        "train_mape_pct": [
            round(mape_percent(y_train_split, ridge_train_pred), 3)
        ],
        "validation_mape_pct": [
            round(mape_percent(y_valid_split, ridge_valid_pred), 3)
        ],
        "train_size": [len(y_train_split)],
        "validation_size": [len(y_valid_split)],
        "notes": [
            "car_id и Предложение исключены; alpha=0.1"
        ],
    }
)

display(best_ridge_holdout_result)

,experiment_id,model,split,train_mape_pct,validation_mape_pct,train_size,validation_size,notes
0,ridge_ohe_log1p_alpha_0_1_holdout_rs42,Ridge + OHE + log1p(target),train_valid_80_20_stratified_qcut10_rs42,2.626,14.194,6672,1668,car_id и Предложение исключены; alpha=0.1


Сохраняю в CSV.

In [29]:
results_path = REPORTS_DIR / "experiment_results.csv"

if results_path.exists():
    experiment_results = pd.read_csv(results_path)
    
    experiment_results = experiment_results.loc[
        experiment_results["experiment_id"]
        != "ridge_ohe_log1p_alpha_0_1_holdout_rs42"
    ]
else:
    experiment_results = pd.DataFrame()

experiment_results = pd.concat(
    [experiment_results, best_ridge_holdout_result],
    ignore_index=True,
)

experiment_results.to_csv(
    results_path,
    index=False,
    encoding="utf-8-sig",
)

display(experiment_results)

,experiment_id,model,split,train_mape_pct,validation_mape_pct,train_size,validation_size,notes
0,ridge_ohe_log1p_alpha_0_1_holdout_rs42,Ridge + OHE + log1p(target),train_valid_80_20_stratified_qcut10_rs42,2.626,14.194,6672,1668,car_id и Предложение исключены; alpha=0.1


сохраняю validation-предсказания Ridge. Это пригодится для анализа ошибок и будущего ансамбля Ridge + CatBoost.

In [30]:
ridge_valid_predictions = pd.DataFrame(
    {
        "car_id": train.loc[y_valid_split.index, "car_id"].to_numpy(),
        "y_true": y_valid_split.to_numpy(),
        "ridge_pred": ridge_valid_pred,
    }
)

ridge_valid_predictions["ape_pct"] = (
    np.abs(
        ridge_valid_predictions["y_true"]
        - ridge_valid_predictions["ridge_pred"]
    )
    / ridge_valid_predictions["y_true"]
    * 100
)

ridge_valid_predictions.to_parquet(
    REPORTS_DIR / "ridge_valid_predictions_alpha_0_1.parquet",
    index=False,
)

Сохраняю Ridge OOF

In [33]:
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

BEST_RIDGE_ALPHA = 0.1

ridge_oof_predictions = pd.Series(
    index=X_ridge_all.index,
    dtype=float,
)

ridge_oof_rows = []

for fold_number, (train_idx, valid_idx) in enumerate(
    cv.split(X_ridge_all, cv_target_bins),
    start=1,
):
    X_fold_train = X_ridge_all.iloc[train_idx]
    X_fold_valid = X_ridge_all.iloc[valid_idx]

    y_fold_train = y.iloc[train_idx]
    y_fold_valid = y.iloc[valid_idx]

    ridge_fold_pipeline = Pipeline(
        steps=[
            ("preprocessor", clone(preprocessor)),
            (
                "model",
                Ridge(
                    alpha=BEST_RIDGE_ALPHA,
                    solver="lsqr",
                ),
            ),
        ]
    )

    ridge_fold_model = TransformedTargetRegressor(
        regressor=ridge_fold_pipeline,
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False,
    )

    ridge_fold_model.fit(
        X_fold_train,
        y_fold_train,
    )

    fold_pred = np.maximum(
        ridge_fold_model.predict(X_fold_valid),
        1,
    )

    ridge_oof_predictions.iloc[valid_idx] = fold_pred

    ridge_oof_rows.append(
        {
            "fold": fold_number,
            "validation_mape_pct": round(
                mape_percent(y_fold_valid, fold_pred),
                3,
            ),
        }
    )

ridge_oof_folds = pd.DataFrame(ridge_oof_rows)

display(ridge_oof_folds)

print(
    "Ridge OOF MAPE:",
    round(mape_percent(y, ridge_oof_predictions), 3),
)

assert ridge_oof_predictions.notna().all()
assert len(ridge_oof_predictions) == len(y)

,fold,validation_mape_pct
0,1,14.822
1,2,14.417
2,3,14.369
3,4,14.567
4,5,14.170


Ridge OOF MAPE: 14.469


In [34]:
ridge_oof = pd.DataFrame(
    {
        "car_id": train.loc[
            ridge_oof_predictions.index,
            "car_id"
        ].to_numpy(),
        "y_true": y.to_numpy(),
        "ridge_pred": ridge_oof_predictions.to_numpy(),
    }
)

assert ridge_oof["car_id"].is_unique
assert ridge_oof["ridge_pred"].notna().all()

ridge_oof_path = (
    REPORTS_DIR
    / "ridge_oof_predictions_alpha_0_1.parquet"
)

ridge_oof.to_parquet(
    ridge_oof_path,
    index=False,
)

print(ridge_oof_path)
display(ridge_oof.head())

C:\temp\shift_ml\reports\ridge_oof_predictions_alpha_0_1.parquet


,car_id,y_true,ridge_pred
0,65e4207d-80c1-47a9-9ce8-51a5e5cfca5c,38812,40566.385555
1,331006ba-098d-4b5b-9f05-3cf9544c30ec,15950,16612.439448
2,43a44ee6-5293-4a42-a04f-faf3bf0967b4,41990,40958.198793
3,383affa3-d8da-4ad0-9bd7-91f48c1c00ac,69900,59670.249223
4,e100c33a-b78c-479d-8edf-5734b2bde7fd,27950,27712.258776


In [35]:
ridge_holdout = pd.DataFrame(
    {
        "car_id": train.loc[y_valid_split.index, "car_id"].to_numpy(),
        "y_true": y_valid_split.to_numpy(),
        "ridge_pred": ridge_valid_pred,
    }
)

ridge_holdout.to_parquet(
    REPORTS_DIR / "ridge_holdout_predictions_alpha_0_1.parquet",
    index=False,
)

display(ridge_holdout.head())

,car_id,y_true,ridge_pred
0,c8308aaa-236b-4d69-b6ed-1656a0da72a7,19990,29376.245745
1,0d0cc3b5-b9a2-4417-92ef-5bf6975467e5,75990,71057.755352
2,ad545ff1-51f5-486c-9293-0c5ee10a7bcd,34485,33790.401383
3,37f7453f-ef82-428d-9bc4-f73d9689b194,6250,5180.029616
4,6a9a3d1a-dc59-4498-8f8d-23c7be368461,21800,23475.926417
